# 21. Baseline — DeepJoin

DeepJoin(Dong et al., VLDB 2023)의 컬럼 임베딩 기반 joinable column 탐색을 FK 추론 평가 프로토콜에 맞춰 재구현했다
(`sentence-transformers/all-MiniLM-L6-v2`). 후보쌍은 방향 없는 쌍으로 대칭화해 CRIM 과 같은 방식으로 평가한다.

- 출력: `workspace/output_deepjoin/deepjoin_ranking_metrics.json`

# DeepJoin Baseline for FK Detection

CRIM_Asymmetric 모델의 비교 baseline으로 사용할 **DeepJoin** (VLDB 2023) 구현.

**파이프라인:**
1. 학습 데이터 (CSV 폴더 + GT JSON) → column contextualization
2. Sentence-BERT 기반 metric learning (Multiple Negatives Ranking Loss)
3. 추론: 학습된 모델로 테스트 DB의 모든 column 쌍 cosine similarity 계산
4. 평가: TP/FP/TN/FN, Precision/Recall/F1 산출

**DeepJoin equi-joinability:**
$$\text{joinability}(Q \to X) = \frac{|Q \cap X|}{|Q|}$$

학습은 column embedding 간 cosine similarity가 이 신호와 일치하도록 PLM을 fine-tuning.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q sentence-transformers

In [ ]:
# === Imports ===
import os
import sys
import json
import random
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sentence_transformers import SentenceTransformer, InputExample, losses, util

warnings.filterwarnings('ignore')

# Google Drive 마운트 (Colab 환경에서만)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
else:
    print("Running locally — Drive mount skipped.")

# 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Torch: {torch.__version__}")

In [ ]:
# === Paths (사용자 수정 필요) ===

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    # Google Drive 경로 (사용자가 Drive에 업로드한 위치에 맞게 수정)
    DRIVE_ROOT     = "/content/drive/MyDrive/crim/0511"
    TRAIN_DATA_DIR = f"{DRIVE_ROOT}/train"
    TRAIN_GT_PATH  = f"{DRIVE_ROOT}/result/ground_truth.json"
    BASE_TEST_DIR  = f"{DRIVE_ROOT}/dataset_mini/bird_data_tables"
    OUTPUT_DIR     = f"{DRIVE_ROOT}/output_deepjoin"
else:
    # 로컬 경로 (기존 유지)
    TRAIN_DATA_DIR = "../data/tpc/train"
    TRAIN_GT_PATH  = "../workspace/result/ground_truth.json"
    BASE_TEST_DIR  = "../workspace/dataset_mini/bird_data_tables"
    OUTPUT_DIR     = "../workspace/output_deepjoin"

DB_NAME       = "toxicology"  # 사용자 수정: 테스트할 DB 이름
DB_NAME       = "thrombosis_prediction"  # 사용자 수정: 테스트할 DB 이름
DB_NAME       = "superhero"  # 사용자 수정: 테스트할 DB 이름
DB_NAME       = "california_schools"  # 사용자 수정: 테스트할 DB 이름
# DB_NAME       = "toxicology_mask"  # 사용자 수정: 테스트할 DB 이름
# DB_NAME       = "thrombosis_prediction_mask"  # 사용자 수정: 테스트할 DB 이름
# DB_NAME       = "superhero_mask"  # 사용자 수정: 테스트할 DB 이름
# DB_NAME       = "california_schools_mask"  # 사용자 수정: 테스트할 DB 이름
TEST_DATA_DIR = f"{BASE_TEST_DIR}/{DB_NAME}"
TEST_GT_PATH  = f"{BASE_TEST_DIR}/{DB_NAME}/ground_truth.json"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "deepjoin_finetuned")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Hyperparameters ===
PLM_MODEL        = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ_LENGTH   = 256
BATCH_SIZE       = 32 if IS_COLAB else 16   # Colab GPU면 32, 로컬은 16
EPOCHS           = 3
LEARNING_RATE    = 2e-5

# 컬럼 직렬화 옵션
SERIALIZATION_MODE = 'title-colname-stat-values'   # DeepJoin 논문 best
NUM_SAMPLE_VALUES  = 30                             # 컬럼당 샘플 개수

# 추론 옵션
SIMILARITY_THRESHOLD = 0.70    # cosine similarity threshold
TOPK_PER_CHILD       = 1       # child 컬럼당 최대 parent 수

print("Config loaded.")
print(f"  IS_COLAB: {IS_COLAB}")
print(f"  DB_NAME: {DB_NAME}")
print(f"  TRAIN_DATA_DIR: {TRAIN_DATA_DIR}")
print(f"  TEST_DATA_DIR: {TEST_DATA_DIR}")
print(f"  PLM: {PLM_MODEL}")
print(f"  MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}, BATCH_SIZE: {BATCH_SIZE}")
print(f"  Serialization mode: {SERIALIZATION_MODE}")
print(f"  Threshold: {SIMILARITY_THRESHOLD}, TopK: {TOPK_PER_CHILD}")

In [ ]:
# === Data Loading (Train Only) ===

from tqdm import tqdm

def load_database_tables(folder_path, db_name="default_db"):
    """
    폴더 구조: folder_path/{table_name}.csv
    Returns: {db_name: {table_name: DataFrame}}
    """
    folder = Path(folder_path)
    db_tables = {db_name: {}}
    
    for csv_file in folder.glob("*.csv"):
        table_name = csv_file.stem
        try:
            df = pd.read_csv(csv_file, low_memory=False)
            # float ID -> Int64 변환 (CRIM 노트북과 동일 로직)
            for col in df.select_dtypes(include='float64').columns:
                non_null = df[col].dropna()
                if len(non_null) > 0 and (non_null % 1 == 0).all():
                    df[col] = df[col].astype('Int64')
            db_tables[db_name][table_name] = df
        except Exception as e:
            print(f"[WARN] skipped {csv_file}: {e}")
    
    return db_tables


def load_fk_pairs(json_path):
    """
    실제 사용되는 3가지 GT 포맷을 모두 정규화하여
    공통 형태 {db_name: {"fk_pairs": [...]}} 로 반환.

    Format A — flat keys (DeepJoin 원본 가정):
      {db_name: {"fk_pairs": [{"child_table":..., "child_column":...,
                                "parent_table":..., "parent_column":...}, ...]}}

    Format B — nested list (CRIM 학습 GT, source/result/ground_truth.json):
      {db_name: {"fk_pairs": [{"child":[t,c], "parent":[t,c]}, ...]}}

    Format C — list of pair strings (BIRD 테스트 GT,
      dataset_mini/.../<db>/ground_truth.json):
      [["table.col", "table.col"], ...]
      → 이 경우 db_name 은 파일 부모 폴더명으로 유추.
    """
    from pathlib import Path as _Path

    with open(json_path, 'r') as f:
        raw = json.load(f)

    # Format C: list at root → wrap with parent folder name
    if isinstance(raw, list):
        db_name = _Path(json_path).parent.name
        pairs = []
        for item in raw:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                pairs.append({'child': item[0], 'parent': item[1]})
            elif isinstance(item, dict):
                pairs.append(item)
        return {db_name: {'fk_pairs': pairs}}

    # Format A/B: dict at root — return as-is (gt_to_set 가 child/parent 키 형태도 처리)
    return raw


def gt_to_set(gt_dict):
    """GT -> (db, c_tab, c_col, p_tab, p_col) tuple set

    지원 포맷:
      A) flat keys      : {'child_table':..., 'child_column':..., 'parent_table':..., 'parent_column':...}
      B) nested list    : {'child': [table, column], 'parent': [table, column]}   ← CRIM 노트북과 동일
      C) string '.' fmt : {'child': 'table.column', 'parent': 'table.column'}
    """
    def _split(v):
        if isinstance(v, (list, tuple)):
            return v[0], v[1]
        if isinstance(v, str) and '.' in v:
            t, c = v.split('.', 1)
            return t, c
        raise ValueError(f"unrecognized child/parent value: {v!r}")

    fk_set = set()
    for db_name, info in gt_dict.items():
        pairs = info.get('fk_pairs', []) if isinstance(info, dict) else info
        for p in pairs:
            if 'child_table' in p:                       # Format A
                key = (db_name,
                       p['child_table'], p['child_column'],
                       p['parent_table'], p['parent_column'])
            elif 'child' in p and 'parent' in p:         # Format B / C
                c_tab, c_col = _split(p['child'])
                p_tab, p_col = _split(p['parent'])
                key = (db_name, c_tab, c_col, p_tab, p_col)
            else:
                raise KeyError(f"FK pair has unknown schema: {list(p.keys())}")
            fk_set.add(key)
    return fk_set


# === 훈련 데이터만 로드 ===
import gc

print("Loading training data...")
train_gt     = load_fk_pairs(TRAIN_GT_PATH)
train_db_names = list(train_gt.keys())
train_tables = {}
for db_name in tqdm(train_db_names, desc="Loading train DBs"):
    train_tables.update(load_database_tables(TRAIN_DATA_DIR, db_name=db_name))
train_fk_set = gt_to_set(train_gt)
gc.collect()

print(f"\n[Train] DBs: {len(train_tables)} | FK pairs: {len(train_fk_set)}")
for db, tabs in train_tables.items():
    n_fk = sum(1 for fk in train_fk_set if fk[0] == db)
    print(f"  {db}: {len(tabs)} tables, {n_fk} FKs")

In [ ]:
# === Column Contextualization (Train Only) ===

def contextualize_column(table_name, col_name, col_series,
                         num_samples=30, mode='title-colname-stat-values'):
    """
    DeepJoin 스타일로 컬럼을 텍스트 시퀀스로 변환.
    
    Modes (DeepJoin 논문):
    - 'colname-values'           : 컬럼명 + 샘플 값
    - 'title-colname-stat'       : 테이블명 + 컬럼명 + 통계
    - 'title-colname-stat-values': 위 모두 (best 성능)
    """
    non_null = col_series.dropna()
    
    if len(non_null) == 0:
        return f"{table_name}. {col_name}. (empty)"
    
    # 샘플 값 추출
    n = min(num_samples, len(non_null))
    samples = non_null.sample(n, random_state=SEED).astype(str).tolist()
    values_str = ", ".join(samples)
    
    # 통계
    n_distinct = int(non_null.nunique())
    n_total    = int(len(non_null))
    n_null     = int(col_series.isna().sum())
    
    is_numeric = pd.api.types.is_numeric_dtype(col_series)
    try:
        avg_len = float(non_null.astype(str).str.len().mean())
    except Exception:
        avg_len = 0.0
    
    if is_numeric:
        stat_str = (f"numeric column. distinct {n_distinct}, total {n_total}, "
                    f"null {n_null}, avg-length {avg_len:.0f}")
    else:
        stat_str = (f"text column. distinct {n_distinct}, total {n_total}, "
                    f"null {n_null}, avg-length {avg_len:.0f}")
    
    if mode == 'colname-values':
        text = f"{col_name}. {values_str}"
    elif mode == 'title-colname-stat':
        text = f"{table_name}. {col_name}. {stat_str}"
    elif mode == 'title-colname-stat-values':
        text = f"{table_name}. {col_name}. {stat_str}. values: {values_str}"
    else:
        text = f"{table_name}. {col_name}. {values_str}"
    
    return text


def build_column_corpus(db_tables, mode):
    """
    Returns: {(db, table, col): text}
    """
    corpus = {}
    for db_name, tables in db_tables.items():
        for table_name, df in tables.items():
            for col_name in df.columns:
                key = (db_name, table_name, col_name)
                text = contextualize_column(table_name, col_name, df[col_name],
                                            num_samples=NUM_SAMPLE_VALUES, mode=mode)
                corpus[key] = text
    return corpus


# === 훈련 corpus 생성 ===
train_corpus = build_column_corpus(train_tables, mode=SERIALIZATION_MODE)
print(f"Train columns: {len(train_corpus)}")
print("\n[Sample serialized column]")
sample_key = list(train_corpus.keys())[0]
print(f"  key: {sample_key}")
print(f"  text: {train_corpus[sample_key][:300]}...")

In [ ]:
# === Training Pair Preparation (MultipleNegativesRankingLoss) ===

def prepare_training_pairs(corpus, fk_set):
    """
    Positive pair (child_text, parent_text) 생성.
    MultipleNegativesRankingLoss 사용 시 batch 내 다른 sample들이 negative로 자동 사용됨.
    """
    pairs = []
    missing = 0
    for fk in fk_set:
        db, c_tab, c_col, p_tab, p_col = fk
        c_key = (db, c_tab, c_col)
        p_key = (db, p_tab, p_col)
        
        if c_key in corpus and p_key in corpus:
            pairs.append(InputExample(texts=[corpus[c_key], corpus[p_key]]))
        else:
            missing += 1
    
    return pairs, missing


train_examples, missing = prepare_training_pairs(train_corpus, train_fk_set)
print(f"Training pairs: {len(train_examples)} (missing: {missing})")

if len(train_examples) == 0:
    raise ValueError("학습 pair가 0개입니다. GT JSON 형식 또는 폴더 경로를 확인하세요.")

In [ ]:
# === Model Loading & Training ===

print(f"Loading PLM: {PLM_MODEL}")
model = SentenceTransformer(PLM_MODEL)
model.max_seq_length = MAX_SEQ_LENGTH
model.to(device)

print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# DataLoader + Loss
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(len(train_dataloader) * EPOCHS * 0.1)
print(f"Starting training: {EPOCHS} epochs, {len(train_dataloader)} steps/epoch, warmup={warmup_steps}")


In [ ]:

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LEARNING_RATE},
    show_progress_bar=True
)

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
model.save(MODEL_SAVE_PATH)
print(f"\nModel saved: {MODEL_SAVE_PATH}")

# === 학습 데이터 메모리에서 삭제 ===
print("\n[Cleaning up training data...]")
del train_tables, train_corpus, train_dataloader, train_examples, train_fk_set
gc.collect()
print("[Training data deleted from memory]")

In [ ]:
# === Test 데이터 로드 (추론 전) ===

from tqdm import tqdm

print("Loading test data...")
test_gt      = load_fk_pairs(TEST_GT_PATH)
test_db_names = list(test_gt.keys())
test_tables = {}
for db_name in tqdm(test_db_names, desc="Loading test DBs"):
    test_tables.update(load_database_tables(TEST_DATA_DIR, db_name=db_name))
test_fk_set  = gt_to_set(test_gt)
gc.collect()

print(f"\n[Test ] DBs: {len(test_tables)} | FK pairs: {len(test_fk_set)}")
for db, tabs in test_tables.items():
    n_fk = sum(1 for fk in test_fk_set if fk[0] == db)
    print(f"  {db}: {len(tabs)} tables, {n_fk} FKs")

# Test corpus 생성
test_corpus = build_column_corpus(test_tables, mode=SERIALIZATION_MODE)
print(f"\nTest columns: {len(test_corpus)}")

# Test 데이터프레임 삭제 (corpus만 유지)
del test_tables
gc.collect()
print("[Test DataFrames deleted, test_corpus retained]\n")

# === Inference: 모든 column pair에 대해 cosine similarity 계산 ===

def predict_fk_pairs(model, test_corpus,
                     threshold=SIMILARITY_THRESHOLD, topk=TOPK_PER_CHILD):
    """
    DB 단위로:
      1. DB 내 모든 컬럼 임베딩
      2. (child, parent) 후보 (다른 테이블끼리) 생성
      3. cosine similarity 계산
      4. Child별 top-k + threshold 적용 → predictions
    
    Returns:
      predictions: set of (db, c_tab, c_col, p_tab, p_col)
      scores_log : DataFrame of all candidate scores
    """
    predictions = set()
    all_scores = []
    
    # test_corpus에서 DB 이름 추출
    test_db_names = sorted(set(k[0] for k in test_corpus.keys()))
    
    for db_name in tqdm(test_db_names, desc="Running inference"):
        # DB 내 컬럼만 추출
        db_keys = [k for k in test_corpus.keys() if k[0] == db_name]
        if len(db_keys) < 2:
            continue
        
        db_texts  = [test_corpus[k] for k in db_keys]
        db_embeds = model.encode(db_texts, convert_to_tensor=True,
                                 show_progress_bar=False, batch_size=64)
        
        # 후보 생성 (방향성 있음, 같은 테이블 제외)
        child_scores = {}
        
        for i, c_key in enumerate(db_keys):
            c_db, c_tab, c_col = c_key
            child_scores[c_key] = []
            
            for j, p_key in enumerate(db_keys):
                if i == j:
                    continue
                p_db, p_tab, p_col = p_key
                if c_tab == p_tab:  # same-table pair 제외
                    continue
                
                sim = float(util.cos_sim(db_embeds[i], db_embeds[j]).item())
                child_scores[c_key].append((sim, p_key))
                
                all_scores.append({
                    'db': db_name,
                    'child_table':  c_tab,
                    'child_column': c_col,
                    'parent_table': p_tab,
                    'parent_column': p_col,
                    'similarity': sim
                })
        
        # Top-k per child + threshold
        for c_key, scored in child_scores.items():
            scored.sort(key=lambda x: -x[0])
            for sim, p_key in scored[:topk]:
                if sim >= threshold:
                    c_db, c_tab, c_col = c_key
                    _, p_tab, p_col = p_key
                    predictions.add((c_db, c_tab, c_col, p_tab, p_col))
    
    scores_df = pd.DataFrame(all_scores).sort_values('similarity', ascending=False)
    return predictions, scores_df


print("Running inference on test data...")
predicted_fks, scores_df = predict_fk_pairs(model, test_corpus)
print(f"\nTotal predictions (threshold={SIMILARITY_THRESHOLD}, topk={TOPK_PER_CHILD}): {len(predicted_fks)}")
print(f"Total candidate scores logged: {len(scores_df)}")

In [ ]:
# === Evaluation: TP/FP/TN/FN, Precision/Recall/F1 ===

def build_candidate_space(corpus):
    """
    DB 별로 가능한 모든 (child, parent) 후보 (방향 있음, 다른 테이블 간) 생성.
    """
    by_db = {}
    for k in corpus.keys():
        by_db.setdefault(k[0], []).append(k)
    
    candidates = set()
    for db_name, keys in by_db.items():
        for c_key in keys:
            for p_key in keys:
                if c_key == p_key or c_key[1] == p_key[1]:
                    continue
                candidates.add((c_key[0], c_key[1], c_key[2], p_key[1], p_key[2]))
    return candidates


def compute_metrics(pred, gt, candidates):
    """단일 set 기준 metrics 계산"""
    tp = pred & gt
    fp = pred - gt
    fn = gt - pred
    tn = candidates - pred - gt
    
    n_tp, n_fp, n_fn, n_tn = len(tp), len(fp), len(fn), len(tn)
    
    precision = n_tp / (n_tp + n_fp) if (n_tp + n_fp) > 0 else 0.0
    recall    = n_tp / (n_tp + n_fn) if (n_tp + n_fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    
    return {
        'TP': n_tp, 'FP': n_fp, 'TN': n_tn, 'FN': n_fn,
        'precision': precision, 'recall': recall, 'f1': f1
    }, tp, fp, fn


candidate_space = build_candidate_space(test_corpus)
print(f"Total candidate space (test): {len(candidate_space)}")

# 전체 평가
overall_metrics, tp_all, fp_all, fn_all = compute_metrics(
    predicted_fks, test_fk_set, candidate_space)

print("\n" + "=" * 60)
print("OVERALL METRICS (across all test DBs)")
print("=" * 60)
print(f"  TP: {overall_metrics['TP']:>5d}")
print(f"  FP: {overall_metrics['FP']:>5d}")
print(f"  TN: {overall_metrics['TN']:>5d}")
print(f"  FN: {overall_metrics['FN']:>5d}")
print(f"  Precision: {overall_metrics['precision']:.4f}")
print(f"  Recall:    {overall_metrics['recall']:.4f}")
print(f"  F1-score:  {overall_metrics['f1']:.4f}")

In [ ]:
# === Per-DB Evaluation (CRIM 논문 비교용) ===

def evaluate_per_db(pred, gt, corpus):
    """DB별 평가 (corpus에서 DB 정보 추출)"""
    results = {}
    
    # corpus에서 DB 이름 추출
    db_names = sorted(set(k[0] for k in corpus.keys()))
    
    for db_name in db_names:
        db_pred = {p for p in pred if p[0] == db_name}
        db_gt   = {p for p in gt   if p[0] == db_name}
        
        # DB 내 후보 공간
        db_keys = [k for k in corpus.keys() if k[0] == db_name]
        db_cands = set()
        for c in db_keys:
            for p in db_keys:
                if c == p or c[1] == p[1]:
                    continue
                db_cands.add((c[0], c[1], c[2], p[1], p[2]))
        
        metrics, _, _, _ = compute_metrics(db_pred, db_gt, db_cands)
        results[db_name] = metrics
    
    return results


per_db_results = evaluate_per_db(predicted_fks, test_fk_set, test_corpus)
df_per_db = pd.DataFrame(per_db_results).T

print("=" * 70)
print("PER-DB METRICS")
print("=" * 70)
print(df_per_db.to_string())

# Macro 평균
macro_p  = df_per_db['precision'].mean()
macro_r  = df_per_db['recall'].mean()
macro_f1 = df_per_db['f1'].mean()
print(f"\nMacro Precision: {macro_p:.4f}")
print(f"Macro Recall:    {macro_r:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")

In [ ]:
# === 상세 결과 저장 및 분석 ===

# 1. 모든 후보 score 저장
scores_path = os.path.join(OUTPUT_DIR, 'deepjoin_all_scores.csv')
scores_df.to_csv(scores_path, index=False)
print(f"All scores saved: {scores_path} ({len(scores_df)} rows)")

# 2. 예측 결과 저장
pred_df = pd.DataFrame(sorted(predicted_fks),
                       columns=['db', 'child_table', 'child_column', 'parent_table', 'parent_column'])
pred_path = os.path.join(OUTPUT_DIR, 'deepjoin_predictions.csv')
pred_df.to_csv(pred_path, index=False)
print(f"Predictions saved: {pred_path} ({len(pred_df)} rows)")

# 3. 메트릭 저장
metrics_path = os.path.join(OUTPUT_DIR, 'deepjoin_metrics.json')
metrics_out = {
    'config': {
        'PLM_MODEL': PLM_MODEL,
        'serialization_mode': SERIALIZATION_MODE,
        'num_sample_values': NUM_SAMPLE_VALUES,
        'threshold': SIMILARITY_THRESHOLD,
        'topk_per_child': TOPK_PER_CHILD,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE
    },
    'overall': overall_metrics,
    'per_db': per_db_results,
    'macro': {'precision': macro_p, 'recall': macro_r, 'f1': macro_f1}
}
with open(metrics_path, 'w') as f:
    json.dump(metrics_out, f, indent=2, default=float)
print(f"Metrics saved: {metrics_path}")

In [ ]:
# === FN (놓친 GT) / FP (잘못 예측) 분석 ===

def fk_similarity(scores_df, fk_tuple):
    """특정 FK 튜플의 score 검색"""
    db, c_tab, c_col, p_tab, p_col = fk_tuple
    m = scores_df[
        (scores_df['db'] == db) &
        (scores_df['child_table']   == c_tab) &
        (scores_df['child_column']  == c_col) &
        (scores_df['parent_table']  == p_tab) &
        (scores_df['parent_column'] == p_col)
    ]
    return m.iloc[0]['similarity'] if len(m) > 0 else None


print("=" * 80)
print(f"FALSE NEGATIVES — 놓친 GT ({len(fn_all)}개)")
print("=" * 80)
for fn in sorted(fn_all):
    sim = fk_similarity(scores_df, fn)
    sim_str = f"sim={sim:.3f}" if sim is not None else "sim=N/A"
    print(f"  [{fn[0]:>22s}] {fn[1]}.{fn[2]} -> {fn[3]}.{fn[4]}  ({sim_str})")

print("\n" + "=" * 80)
print(f"TOP-20 FALSE POSITIVES (잘못 예측, similarity 높은 순)")
print("=" * 80)
fp_with_sim = [(fp, fk_similarity(scores_df, fp)) for fp in fp_all]
fp_with_sim = [(fp, s) for fp, s in fp_with_sim if s is not None]
fp_with_sim.sort(key=lambda x: -x[1])
for fp, sim in fp_with_sim[:20]:
    print(f"  [{fp[0]:>22s}] {fp[1]}.{fp[2]} -> {fp[3]}.{fp[4]}  (sim={sim:.3f})")

In [ ]:
# === Threshold sweep: 다른 threshold에서의 성능 ===

print("Threshold sweep:")
print(f"{'threshold':>10} | {'TP':>4} | {'FP':>5} | {'FN':>4} | {'Prec':>6} | {'Rec':>6} | {'F1':>6}")
print("-" * 60)

best_f1 = 0.0
best_thr = SIMILARITY_THRESHOLD

# test_corpus에서 DB 이름 추출
test_db_names_sweep = sorted(set(k[0] for k in test_corpus.keys()))

for thr in [0.3, 0.4, 0.5, 0.6, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    # 다시 prediction 생성 (cell 8 로직과 동일, topk=1 고정)
    pred_t = set()
    
    for db_name in test_db_names_sweep:
        db_keys = [k for k in test_corpus.keys() if k[0] == db_name]
        if len(db_keys) < 2:
            continue
        
        db_subset = scores_df[scores_df['db'] == db_name]
        
        # child별 top-1
        for c_key in db_keys:
            c_db, c_tab, c_col = c_key
            m = db_subset[
                (db_subset['child_table']  == c_tab) &
                (db_subset['child_column'] == c_col)
            ]
            if len(m) == 0:
                continue
            top1 = m.sort_values('similarity', ascending=False).iloc[0]
            if top1['similarity'] >= thr:
                pred_t.add((c_db, c_tab, c_col,
                            top1['parent_table'], top1['parent_column']))
    
    m_t, _, _, _ = compute_metrics(pred_t, test_fk_set, candidate_space)
    print(f"{thr:>10.2f} | {m_t['TP']:>4d} | {m_t['FP']:>5d} | {m_t['FN']:>4d} | "
          f"{m_t['precision']:>6.3f} | {m_t['recall']:>6.3f} | {m_t['f1']:>6.3f}")
    if m_t['f1'] > best_f1:
        best_f1 = m_t['f1']
        best_thr = thr

print(f"\nBest threshold: {best_thr} (F1={best_f1:.4f})")

## 최종 결과 요약

이 노트북의 결과는 `{OUTPUT_DIR}` 폴더에 저장:

| 파일 | 내용 |
|---|---|
| `deepjoin_finetuned/` | Fine-tuned SBERT 모델 (재사용 가능) |
| `deepjoin_all_scores.csv` | 모든 후보 (child, parent) 쌍의 cosine similarity |
| `deepjoin_predictions.csv` | Threshold + top-k 적용된 FK 예측 |
| `deepjoin_metrics.json` | overall / per-DB / macro 메트릭 + config |

## CRIM_Asymmetric과의 비교 시 유의점

1. **공정한 비교를 위해**: CRIM의 평가 DB(toxicology, california_schools, superhero, thrombosis_prediction)와 동일하게 평가
2. **Threshold sweep**: DeepJoin 최적 threshold를 찾아 사용 (Cell 13 결과 활용)
3. **예상 차이점**:
   - **Text 컬럼이 풍부한 DB** (california_schools): DeepJoin이 PLM 기반 의미 인코딩으로 어느 정도 동작
   - **Numeric ID 중심 DB** (toxicology, superhero): DeepJoin 취약, CRIM이 우위 예상
4. **논문 서술 포인트**: "DeepJoin's PLM-based encoding excels on text-rich FK pairs but struggles with numeric ID columns that dominate real-world FK relationships. CRIM_Asymmetric's column-statistics-based approach is type-agnostic and explicitly models the asymmetric cardinality structure (uq_child << uq_parent), achieving superior performance especially on numeric-ID-heavy databases."

## Troubleshooting

- **GT JSON 형식 다를 시**: Cell 4의 `load_fk_pairs`/`gt_to_set` 함수 수정
- **CUDA OOM**: `BATCH_SIZE` 8 또는 4로 축소, `MAX_SEQ_LENGTH` 128로 축소
- **F1=0**: Cell 13의 threshold sweep으로 적정 threshold 탐색
- **학습 데이터 부족**: epochs를 5~10으로 증가, 또는 fine-tuning 생략하고 사전학습 모델 그대로 사용 (Cell 7 skip)

In [ ]:
# ── PR-AUC + Recall@K (CRIM / Valentine / EmbDI 와 동일 프로토콜) ────
# 임계값-독립 ranking 지표로 4개 모델을 한 표에 놓고 비교
#   PR-AUC          : 극도로 불균형한 FK 탐지에 적합
#   Recall@K (K=|GT|): Valentine SIGMOD 2021 표준 (K=|GT| → P@K=R@K=F1@K)
#
# DeepJoin은 directional (child→parent) score 출력 →
# CRIM/Valentine과 호환하려면 **frozenset 기반 undirected**로 대칭화 필요.
# 같은 unordered 컬럼쌍의 양방향 score 중 max 를 사용.

from sklearn.metrics import average_precision_score
import numpy as np
import json
import os

# ── DB 단위로 score_map / gt_canonical / candidate_set 구성 ───────
def build_per_db_artifacts(scores_df, candidate_space, test_fk_set):
    """
    DB별로:
      - cand_frozensets : 후보 unordered 쌍 집합 (frozenset)
      - gt_canonical    : GT unordered 쌍 집합 (frozenset)
      - score_map       : frozenset -> max(similarity) over both directions
    """
    per_db = {}
    for db in sorted(set(k[0] for k in candidate_space)):
        # 후보 (undirected)
        cand_fs = set()
        for (d, ct, cc, pt, pc) in candidate_space:
            if d == db:
                cand_fs.add(frozenset({(ct, cc), (pt, pc)}))

        # GT (undirected)
        gt_fs = set()
        for (d, ct, cc, pt, pc) in test_fk_set:
            if d == db:
                gt_fs.add(frozenset({(ct, cc), (pt, pc)}))

        # score_map (양방향 max)
        sub = scores_df[scores_df['db'] == db]
        score_map = {}
        for _, row in sub.iterrows():
            pair = frozenset({(row['child_table'], row['child_column']),
                              (row['parent_table'], row['parent_column'])})
            s = float(row['similarity'])
            score_map[pair] = max(score_map.get(pair, -np.inf), s)

        per_db[db] = {
            'cand_frozensets': cand_fs,
            'gt_canonical':    gt_fs,
            'score_map':       score_map,
        }
    return per_db


per_db_artifacts = build_per_db_artifacts(scores_df, candidate_space, test_fk_set)

# ── DB별 PR-AUC / Recall@K 계산 ─────────────────────────────────
results_ranking = {}

print(f"\n{'='*80}")
print(f"[DeepJoin 임계값-독립 평가 — CRIM 호환 프로토콜]")
print(f"{'='*80}")

for db, art in per_db_artifacts.items():
    cand_list  = sorted(art['cand_frozensets'], key=lambda p: sorted(p))
    gt_set     = art['gt_canonical']
    score_map  = art['score_map']

    K = len(gt_set)
    total = len(cand_list)

    # 후보에 score가 없는 경우는 score=0 (방향 두 개 다 missing은 거의 없음)
    y_true  = np.array([1 if p in gt_set else 0 for p in cand_list])
    y_score = np.array([score_map.get(p, 0.0) for p in cand_list])

    # PR-AUC
    pr_auc = (average_precision_score(y_true, y_score)
              if y_true.sum() > 0 else float('nan'))

    # Recall@K
    if K > 0 and K <= total:
        top_k_idx = np.argsort(-y_score, kind='stable')[:K]
        tp_at_K = int(y_true[top_k_idx].sum())
        recall_at_K = tp_at_K / K
    else:
        tp_at_K, recall_at_K = 0, float('nan')

    # Recall@3K
    if K > 0 and 3*K <= total:
        top_3k_idx = np.argsort(-y_score, kind='stable')[:3*K]
        recall_at_3K = int(y_true[top_3k_idx].sum()) / K
    else:
        recall_at_3K = float('nan')

    results_ranking[db] = {
        'pr_auc':       pr_auc,
        'recall_at_K':  recall_at_K,
        'recall_at_3K': recall_at_3K,
        'tp_at_K':      tp_at_K,
        'n_gt':         K,
        'total_pairs':  total,
        # PR curve 그리기용 raw 데이터
        'y_true':       y_true.astype(int).tolist(),
        'y_score':      [round(float(s), 6) for s in y_score],
    }

    r3k = (f"{recall_at_3K:.4f}" if not np.isnan(recall_at_3K) else '   N/A')
    print(f"\n[{db}]")
    print(f"  후보쌍(undirected) : {total}  |  GT : {K}")
    print(f"  PR-AUC             : {pr_auc:.4f}")
    print(f"  Recall@{K:<3}         : {recall_at_K:.4f}  ({tp_at_K}/{K})")
    print(f"  Recall@{3*K:<3}         : {r3k}")

# ── 요약 테이블 ────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"[DeepJoin DB별 요약]")
print(f"{'='*80}")
print(f"{'DB':<28} {'PR-AUC':>8} {'R@K':>8} {'R@3K':>8} {'|GT|':>5} {'#pairs':>8}")
print('-' * 80)
for db, m in results_ranking.items():
    r3k = f"{m['recall_at_3K']:.4f}" if not np.isnan(m['recall_at_3K']) else '  N/A'
    print(f"{db:<28} {m['pr_auc']:>8.4f} {m['recall_at_K']:>8.4f} "
          f"{r3k:>8} {m['n_gt']:>5} {m['total_pairs']:>8}")

# 매크로 평균
valid_pr  = [m['pr_auc']      for m in results_ranking.values() if not np.isnan(m['pr_auc'])]
valid_rk  = [m['recall_at_K'] for m in results_ranking.values() if not np.isnan(m['recall_at_K'])]
macro_pr  = sum(valid_pr)/len(valid_pr) if valid_pr else float('nan')
macro_rk  = sum(valid_rk)/len(valid_rk) if valid_rk else float('nan')
print('-' * 80)
print(f"{'MACRO AVG':<28} {macro_pr:>8.4f} {macro_rk:>8.4f}")

# ── 저장 (CRIM / Valentine 과 동일 스키마) ─────────────────────────
# 이전에 다른 DB 로 실행한 결과를 유지하기 위해 기존 파일을 읽고 per_db 만 병합.
ranking_output_path = os.path.join(OUTPUT_DIR, 'deepjoin_ranking_metrics.json')

# 1) 기존 결과 로드 (있으면)
if os.path.exists(ranking_output_path):
    with open(ranking_output_path, 'r') as f:
        existing = json.load(f)
    merged_per_db = dict(existing.get('per_db', {}))
    print(f"  [기존 결과 로드] {ranking_output_path}")
    print(f"  기존 DBs: {list(merged_per_db.keys())}")
else:
    merged_per_db = {}
    print(f"  [신규 생성] {ranking_output_path}")

# 2) 이번 실행의 per_db 결과로 덮어쓰기/추가 (같은 DB 재실행 시 최신값으로 갱신)
new_entries = {
    db: {
        'pr_auc':       round(m['pr_auc'], 4),
        'recall_at_K':  round(m['recall_at_K'], 4),
        'recall_at_3K': (round(m['recall_at_3K'], 4)
                         if not np.isnan(m['recall_at_3K']) else None),
        'tp_at_K':      m['tp_at_K'],
        'n_gt':         m['n_gt'],
        'total_pairs':  m['total_pairs'],
        # PR curve 그리기용 raw 데이터
        'y_true':       m.get('y_true'),
        'y_score':      m.get('y_score'),
    } for db, m in results_ranking.items()
}
merged_per_db.update(new_entries)
print(f"  이번 실행 DBs: {list(new_entries.keys())}")
print(f"  병합 후 전체 DBs: {list(merged_per_db.keys())}")

# 3) macro_avg 는 병합된 모든 DB 기준으로 재계산
all_pr_vals = [v['pr_auc']      for v in merged_per_db.values()
               if v.get('pr_auc') is not None]
all_rk_vals = [v['recall_at_K'] for v in merged_per_db.values()
               if v.get('recall_at_K') is not None]
merged_macro = {
    'pr_auc':      round(sum(all_pr_vals)/len(all_pr_vals), 4) if all_pr_vals else None,
    'recall_at_K': round(sum(all_rk_vals)/len(all_rk_vals), 4) if all_rk_vals else None,
    'n_dbs':       len(merged_per_db),
}

# 4) 저장
with open(ranking_output_path, 'w') as f:
    json.dump({
        'model': 'DeepJoin_VLDB2023',
        'per_db': merged_per_db,
        'macro_avg': merged_macro,
        'note': ('Symmetrized (frozenset) for fair comparison with '
                 'CRIM/Valentine; max(similarity) over both directions. '
                 'per_db is incrementally merged across DB-by-DB runs.'),
    }, f, indent=2)
print(f"\n[저장] {ranking_output_path}")
print(f"  병합된 DB 수: {merged_macro['n_dbs']}")
print(f"  macro PR-AUC : {merged_macro['pr_auc']}")
print(f"  macro R@K    : {merged_macro['recall_at_K']}")


In [ ]:
# ── Classic metrics (TP/FP/FN/TN, Precision/Recall/F1) 저장 ─────────
# per_db_results (이미 evaluate_per_db 가 계산) 를 _classic 파일로 저장.
# 다른 DB 로 재실행 시 결과 누적 (per_db merge).

import json
import os

classic_path = os.path.join(OUTPUT_DIR, 'deepjoin_ranking_metrics_classic.json')

# 1) 이번 실행의 per-DB classic 메트릭
this_run_per_db = {}
for db, m in per_db_results.items():
    this_run_per_db[db] = {
        'tp':        int(m.get('TP', 0)),
        'fp':        int(m.get('FP', 0)),
        'fn':        int(m.get('FN', 0)),
        'tn':        int(m.get('TN', 0)),
        'precision': round(float(m.get('precision', 0.0)), 4),
        'recall':    round(float(m.get('recall', 0.0)), 4),
        'f1':        round(float(m.get('f1', 0.0)), 4),
    }

# 2) 기존 파일 로드 → merge (다른 DB 결과 보존)
if os.path.exists(classic_path):
    with open(classic_path) as f:
        existing = json.load(f)
    merged = dict(existing.get('per_db', {}))
else:
    merged = {}
merged.update(this_run_per_db)

# 3) macro 평균 재계산
def _macro(key):
    vals = [v[key] for v in merged.values()]
    return round(sum(vals) / len(vals), 4) if vals else None

macro = {
    'precision': _macro('precision'),
    'recall':    _macro('recall'),
    'f1':        _macro('f1'),
    'n_dbs':     len(merged),
}

with open(classic_path, 'w') as f:
    json.dump({
        'model':     'DeepJoin_VLDB2023',
        'threshold': SIMILARITY_THRESHOLD,
        'per_db':    merged,
        'macro_avg': macro,
    }, f, indent=2)

print(f"[저장] {classic_path}")
print(f"   병합된 DB 수: {len(merged)}  |  macro F1: {macro['f1']}")